# Run a WSM teacher and student trial

Codex writes Python edits against a fresh pinned SmolLM2 target on Modal. Public evaluations guide the edits. An independent hidden grader supplies the final reward, then Spindle trains a separate Qwen3-4B student from the recorded teacher text.

**Run all is offline by default.** It reads the checked-in verification receipt and checks the installed CLI. No model download, credential lookup, Modal request, or training occurs until you explicitly enable the live cell.

From the repository root, start Jupyter with `uv run --project examples/notebooks jupyter lab examples/notebooks/wsm_trial.ipynb`. Select the Python 3 kernel from this environment. See [the notebook setup guide](README.md) for pinned source staging, private credentials, and the marimo version.


In [ ]:
import json
import math
import os
from pathlib import Path
import subprocess
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "scripts/spindle_poc.py").is_file())
NOTEBOOKS = ROOT / "examples/notebooks"
receipt = json.loads((ROOT / "docs/design/trial-verification.json").read_text())


In [ ]:
teacher = receipt["checks"]["live_codex_teacher"]
student = receipt["checks"]["live_spindle_student"]
print({
    "teacher": teacher["teacher"], "teacher_turns": teacher["turns"],
    "target": "SmolLM2-135M-Instruct", "student": student["student"],
    "student_update_verified": student["optimizer_update_successful"] == 1,
    "student_checkpoint_saved": student["checkpoint_saved"],
    "prior_cumulative_reserved_usd": student["cumulative_reserved_usd"],
    "cloud_tasks_remaining": student["cleanup"]["remaining_tasks"],
})


In [ ]:
cli = subprocess.run([sys.executable, str(ROOT / "scripts/spindle_poc.py"), "--help"],
                     cwd=ROOT, capture_output=True, text=True, check=True)
assert "--prior-reserved-usd" in cli.stdout
print("Bounded owner CLI available. No cloud requests were sent.")


## Prepare a live run

Use the commands in [README.md](README.md) to install the live extra with `uv`, authenticate Modal, and stage the pinned source with `stage_smollm`. Run the CLI proxy on the same host as this notebook. Keep private credential JSON files outside the repository; never paste keys or private model routes into cells.

Set the `WSM_NOTEBOOK_*` environment variables before starting Jupyter, then restart the kernel after changing them. `WSM_NOTEBOOK_PRIOR_RESERVED_USD` has no default. The previous proof reserved US$9.927083 of its US$10 allocation, so that allocation cannot fund another full trial. Use the current cumulative total for the allocation you are authorized to spend. Do not reset it to zero to bypass this check.

The live cell invokes the existing bounded owner in a separate process. The owner deploys the task-owned Spindle apps, runs the teacher trial, grades it, trains the student, and confirms cloud cleanup. Its independent watchdog also handles notebook interruption. Full logs stay on disk rather than in notebook outputs. A new output directory is required. Uncertain training updates are never automatically retried.


In [ ]:
config = {
    "model_source": os.environ.get("WSM_NOTEBOOK_MODEL_SOURCE", ""),
    "api_key_file": os.environ.get("WSM_NOTEBOOK_API_KEY_FILE", ""),
    "editor_config_file": os.environ.get("WSM_NOTEBOOK_EDITOR_CONFIG_FILE", ""),
    "output": os.environ.get("WSM_NOTEBOOK_OUTPUT", str(ROOT / "runs/notebook-trial")),
    "prior_reserved_usd": os.environ.get("WSM_NOTEBOOK_PRIOR_RESERVED_USD"),
    "live": os.environ.get("WSM_NOTEBOOK_LIVE") == "1",
}
print({"live_enabled": config["live"], "configured_inputs": {
    key: bool(config[key]) for key in ("model_source", "api_key_file", "editor_config_file")}})


In [ ]:
if not config["live"]:
    print("Offline mode. Set WSM_NOTEBOOK_LIVE=1 only for an authorized live run.")
else:
    prior = config["prior_reserved_usd"]
    if prior is None:
        raise ValueError("Set WSM_NOTEBOOK_PRIOR_RESERVED_USD to the current cumulative reservation")
    prior = float(prior)
    if not math.isfinite(prior) or prior < 0:
        raise ValueError("Prior reservation must be finite and nonnegative")
    # Full default resource windows, two image builds, and artifact egress allowance.
    required = 1.0 + 5 * 600 / 3600 + 8 * 1200 / 3600 + 0.1
    if prior + required > 10:
        raise ValueError("Insufficient remaining US$10 reservation for a new complete trial")
    for field in ("model_source", "api_key_file", "editor_config_file"):
        if not config[field] or not Path(config[field]).expanduser().exists():
            raise ValueError("Configure an existing " + field)
    output = Path(config["output"]).expanduser().absolute()
    if output.exists():
        raise FileExistsError("Use a fresh output directory; existing results are never rerun")
    command = [
        "uv", "run", "--project", str(NOTEBOOKS), "--locked", "--extra", "live", "python",
        str(ROOT / "scripts/spindle_poc.py"),
        "--model-source", str(Path(config["model_source"]).expanduser().absolute()),
        "--grader-python", str(NOTEBOOKS / ".venv/bin/python"),
        "--api-key-file", str(Path(config["api_key_file"]).expanduser().absolute()),
        "--editor-config-file", str(Path(config["editor_config_file"]).expanduser().absolute()),
        "--prior-reserved-usd", str(prior), "--output", str(output),
    ]
    output.parent.mkdir(parents=True, exist_ok=True)
    log_path = output.parent / (output.name + ".owner.log")
    with log_path.open("x") as log:
        owner = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
                                 stdin=subprocess.DEVNULL, start_new_session=True)
    try:
        returncode = owner.wait()
    except KeyboardInterrupt:
        import signal
        owner.send_signal(signal.SIGTERM)
        raise RuntimeError("Stop requested. The owner and watchdog finish cloud cleanup. Inspect the local log.") from None
    if returncode:
        raise RuntimeError("The owner failed. Inspect the private owner log and proof.json before any retry.")
    proof = json.loads((output / "proof.json").read_text())
    print({"state": proof["state"], "reserved_usd": proof["reserved_usd"],
           "cleanup_confirmed": proof["cleanup"]["confirmed"],
           "remaining_tasks": proof["cleanup"]["remaining_tasks"]})


## Inspect a completed run

The live cell prints only the overall state, cumulative reservation, and cleanup status. To read public history in another terminal, use `uv run --project examples/notebooks --extra live wsm trial view runs/notebook-trial/run/trial`. Add `--follow` while the trial is running. Use `wsm trial weights` on a retained safetensors checkpoint to inspect tensor names and numerical values.

`proof.json`, `run/trial/grade.json`, and `run/training/receipt.json` retain the final grade and student update evidence. Keep raw logs and teacher traces private. Clear outputs before sharing a notebook. The resource allowances are reservations, not a provider-enforced billing cap or a reconciled invoice.
